---

# Demo for the PLRE v1.0.0

A driver for demonstrating the Propositional Logic Reasoning Engine (PLRE) v1.0.0.

---

## Imports

In [1]:
import os
import sys

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# Specify whether you have installed the PLRE in your
# Python environment. If you have not done so, don't
# worry, we handle that case.
use_installed_package = False

if use_installed_package:
    pass
else: 
    plre_parent_dir = os.path.abspath('..')
    if not os.path.exists(plre_parent_dir):
        print('Error obtaining PLRE parent directory')
    sys.path.append(plre_parent_dir)
    print(sys.path[-1])

/Users/dave/Library/Mobile Documents/com~apple~CloudDocs/City_PhD/PLRE - Propositional Logic Reasoning Engine/PLRE-Dev-2


In [4]:
from antlr4 import *
from plre.CNFVisitorA import CNFVisitorA
import plre.plre_utils as pu

## Input

A text file containing specifications of propositional logic formulae, expressed in conjunctive normal form (CNF), with respect to some set of propositional logic symbols.

In [5]:
input_file = 'demo1_cnf_expressions.txt'

## Get a set of CNF expressions

Obtain a set of CNF expressions from the input file

In [6]:
expressions = pu.get_cnf_expressions(input_file)

print(f'Number of CNF expressions obtained: {len(expressions)}')

Number of CNF expressions obtained: 17


In [7]:
for idx, expr in enumerate(expressions):
    print(idx+1)
    print(expr)
    print()

1
A

2
(A)

3
!A

4
(!A)

5
(A | B)

6
(A | B | C)

7
(A | !B)

8
(A | !B | C | !D)

9
A & B

10
(A) & (B)

11
A & !B & C

12
!A & B & !C

13
A & (B | C)

14
(A | B) & (C | !D)

15
A & 
B & 
C

16
(A | B | !C) & 
(!B | C | !D)

17
(A | B | !C) & 
(!B | C | !D) & 
(!A | D)



## Parse the CNF expressions

For each CNF expression obtain a parse tree and the parser that generated it

In [8]:
# parse the CNF expressions and obtain a list of parse trees, one tree per expression;
# also capture the parser that generated the tree in case we need it
# [(tree1,parser1), (tree2,parser2), ..., (treeN,parserN)]
trees_and_parsers = [pu.parse_cnf(expr) for expr in expressions]

# isolate the parse trees alone; filter out any trees with
# a value None due to syntax errors having been encountered
trees = [pair[0] for pair in trees_and_parsers if pair[0]]

## Specify a set of propositional symbols

The set of propositional symbols used in the CNF expressions to be processed.

In [9]:
propSymbolSet = ['A', 'B', 'C', 'D']

## Specify a truth-value assignment

A truth-value assignment maps each propositional symbol to a bivalent truth value: True or False.

For convenience, the PLRE allows a truth-value assignment to be specified simply by listing the symbols assigned a value True. All other symbols are assigned value False, by implication.

In [10]:
# list the propositional symbols assigned value True
truthValueAssignment = ['A', 'D']

## Evaluate the truth values of the CNF expressions

In [11]:
# instantiate the PLRE's custom visitor class
visitor = CNFVisitorA(propSymbolSet, truthValueAssignment, verbose=False)

In [12]:
# visit the parse tree of each CNF expression and obtain its 
# truth value, given the truth-value assignment specified earlier
expression_truthValues = [visitor.visit(tree) for tree in trees]

In [13]:
# report propositional symbol usage within the CNF expressions
# processed from the input file
print(f'number of symbols in symbol set: {len(visitor.propSymbolSet)}')
print(f'propSymbolSet: {visitor.propSymbolSet}')
print()
print(f'number of unique symbols used in CNF expressions: {sum(visitor.propSymbolUsage)}')
print(f'propSymbolUsage: {visitor.propSymbolUsage}')

number of symbols in symbol set: 4
propSymbolSet: ['A', 'B', 'C', 'D']

number of unique symbols used in CNF expressions: 4
propSymbolUsage: [True, True, True, True]


In [14]:
# examine the truth values of the CNF expressions found in the input file,
# for the given truth-value assignment
expression_truthValues

[True,
 True,
 False,
 False,
 True,
 True,
 True,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 True]

## Analysis

In [15]:
# Show the number of CNF expressions that evaluated True vs False
print(f'Number of CNF expressions: {len(expression_truthValues)}')
width = len(str(len(expression_truthValues)))
print(f'Number evaluated True    : {str(sum(expression_truthValues)).rjust(width)}')
print(f'Number evaluated False   : {str(len(expression_truthValues) - sum(expression_truthValues)).rjust(width)}')

Number of CNF expressions: 17
Number evaluated True    :  8
Number evaluated False   :  9


In [16]:
# Show the CNF expressions that evaluated True
indices_of_expressions_evaluated_true = [idx for idx, tv in enumerate(expression_truthValues) if tv]
indices_of_expressions_evaluated_true

[0, 1, 4, 5, 6, 7, 15, 16]

In [17]:
# Show the CNF expressions that evaluated False
indices_of_expressions_evaluated_false = [idx for idx, tv in enumerate(expression_truthValues) if not tv]
indices_of_expressions_evaluated_false

[2, 3, 8, 9, 10, 11, 12, 13, 14]

In [18]:
# Show the truth value of the conjunction of the CNF expression truth values
expression_truthValues_cum = True
for expr_tv in expression_truthValues:
    expression_truthValues_cum = expression_truthValues_cum and expr_tv

print(f'The truth value of the conjunction of CNF expression truth values: {expression_truthValues_cum}')

The truth value of the conjunction of CNF expression truth values: False
